In [ ]:
# Ensure Internet toggle on the right side panel is switched ON!
!nvidia-smi

In [ ]:
%%bash
# 1. Clean up any corrupted directory versions cleanly
rm -rf /kaggle/working/spec-fastgs

# 2. Recursively clone using the correct branch name structure
git clone --recursive -b main https://github.com/0Nguyen0Cong0Tuan0/thesis-all.git /kaggle/working/spec-fastgs

# 3. Verify the layout structure
echo "📂 Verifying submodules directory contents:"
ls -la /kaggle/working/spec-fastgs/spec-fastgs/submodules/

In [ ]:
%%bash
# Create the datasets directory inside your workspace and copy files over from input mount
mkdir -p /kaggle/working/spec-fastgs/spec-fastgs/datasets
cp -r /kaggle/input/datasets/nctuan/spec-fastgs-datasets/datasets /kaggle/working/spec-fastgs/spec-fastgs/datasets/

In [ ]:
%%bash
# Clean existing conda toolchain directories safely
rm -rf /opt/conda

# Reinstall isolated Miniconda (Python 3.10)
wget -q https://repo.anaconda.com/miniconda/Miniconda3-py310_23.11.0-1-Linux-x86_64.sh
bash Miniconda3-py310_23.11.0-1-Linux-x86_64.sh -b -p /opt/conda

# Activate custom installation
source /opt/conda/bin/activate

# Install compiler dependencies and CUDA Toolkit 11.7 matching constraints
/opt/conda/bin/conda install -y -c conda-forge cudatoolkit-dev=11.7 gcc_linux-64=11 gxx_linux-64=11

# Export local paths
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH

echo "----- NVCC -----"
nvcc --version
echo "----- PTXAS -----"
ptxas --version

In [ ]:
%%bash
/opt/conda/bin/pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

In [ ]:
%%bash
/opt/conda/bin/python - << 'EOF'
import torch
print("Torch version:", torch.__version__)
print("CUDA back-end:", torch.version.cuda)
print("GPU Available:", torch.cuda.is_available())
EOF

In [ ]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

# 1. diff-gaussian-rasterization
cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 2. simple-knn
cd "../simple-knn"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 3. fused-ssim
cd "../fused-ssim"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

In [ ]:
%%bash
/opt/conda/bin/python - << 'EOF'
import diff_gaussian_rasterization_fastgs
import simple_knn
import fused_ssim
print("✅ FastGS CUDA extensions compiled and loaded successfully!")
EOF

In [ ]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
/opt/conda/bin/python setup.py bdist_wheel

cd "../simple-knn"
/opt/conda/bin/python setup.py bdist_wheel

cd "../fused-ssim"
/opt/conda/bin/python setup.py bdist_wheel

In [ ]:
%%bash
SRC="/kaggle/working/spec-fastgs/spec-fastgs/submodules"
DEST="/kaggle/working/fastgs_wheels_py310"

mkdir -p "$DEST"
find "$SRC" -name "*.whl" -exec cp {} "$DEST" \;

echo "✨ Wheels safely compiled and extracted to: $DEST"
ls -la "$DEST"

In [ ]:
%%bash
/opt/conda/bin/pip uninstall -y numpy
/opt/conda/bin/pip install "numpy<2" plyfile websockets tqdm

In [ ]:
%%bash
/opt/conda/bin/python -c "import fused_ssim; import diff_gaussian_rasterization_fastgs; import plyfile; print('🎉 All systems functional and ready for execution!')"

In [ ]:
%%bash
# ============================================================
# RUN R7 (v3.0) — SPECULAR-AWARE DENSIFICATION (CVPR contribution)
# = R3 residual specular loss + --spec_densify (residual-decomposition vote:
# don't-fake + specular-deficit allocation). Disk-free, no prereq cell.
# Outputs -> ./output/counter_r7
# ============================================================
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH

cd /kaggle/working/spec-fastgs/spec-fastgs

# 1. Dataset layout guard (idempotent)
if [ -d "./datasets/datasets" ]; then
    echo "📦 Re-aligning dataset file structure..."
    mv ./datasets/datasets/* ./datasets/
    rm -rf ./datasets/datasets
fi

# 2. Kick off R7. Watch the banner for 'code: v3.0-...' and 'spec_densify=True (w=0.5)'.
#    The stale-code guard aborts with a 'git pull' message if the checkout predates v3.0.
bash run_spec-fastgs_big_r7.sh


In [ ]:
import shutil, os
# Archive ONLY the R7 output (one run per session — keeps the zip small and avoids
# filling the disk, which is what killed the multi-run Version 28).
src = '/kaggle/working/spec-fastgs/spec-fastgs/output/counter_r7'
out = '/kaggle/working/spec_fastgs_output_r7'
if os.path.isdir(src):
    shutil.make_archive(out, 'zip', src)
    print('archived:', out + '.zip', round(os.path.getsize(out + '.zip')/1e6, 1), 'MB')
else:
    print('no counter_r7 output found at', src)
